In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "eaf7b210",
   "metadata": {},
   "outputs": [],
   "source": [
    "%pylab inline\n",
    "import h5py\n",
    "import scipy\n",
    "import seaborn as sns\n",
    "import pandas as pd"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 24,
   "id": "6831ea15",
   "metadata": {},
   "outputs": [],
   "source": [
    "fname=\"./230905_for_dan_no_sims_vor___icd_only/AO001R_inferior_1.5_S.hdf5\"\n",
    "\n",
    "# These help look inside the hdf5 file: (not needed)\n",
    "def h5py_dataset_iterator(g, prefix=''):\n",
    "    for key, item in g.items():\n",
    "        path = '{}/{}'.format(prefix, key)\n",
    "        if isinstance(item, h5py.Dataset): # test for dataset\n",
    "            yield (path, item)\n",
    "        elif isinstance(item, h5py.Group): # test for group (go down)\n",
    "            yield from h5py_dataset_iterator(item, path)\n",
    "\n",
    "with h5py.File(fname, 'r') as f:\n",
    "    for (path, dset) in h5py_dataset_iterator(f):\n",
    "        #print(path, dset)\n",
    "        pass"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "deb8aaf8",
   "metadata": {},
   "outputs": [],
   "source": [
    "f1=h5py.File(fname,'r')\n",
    "dset=f1['/input_data/cone_coord']\n",
    "coords=np.array(dset)\n",
    "plt.scatter(coords[:,0],coords[:,1] )"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d286cca1",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compute NN distances from matrix of coords\n",
    "def compute_nn(coords):\n",
    "    nn_dist=np.zeros( len(coords))\n",
    "    for ncoord,coord1 in enumerate(coords):\n",
    "        dist1=np.sqrt( np.sum( (coords-coord1)**2, axis=1) )\n",
    "        nn=np.min( dist1[dist1>0] )\n",
    "        nn_dist[ncoord]=nn\n",
    "    return nn_dist\n",
    "\n",
    "# Make the cumulative count of neighbor distances (not binned, based only on the actual x values)\n",
    "def make_stats(coords):\n",
    "    nns = np.sort(compute_nn(coords) )\n",
    "    nn_unique = np.unique(nns)\n",
    "    nn_unique_count = [np.sum(nn1==nns) for nn1 in nn_unique]\n",
    "    pos_count = np.array( list(zip(nn_unique, nn_unique_count) ) ) # combine the two arrays into a 2d np.array\n",
    "    cum_props=np.cumsum(pos_count[:,1])/coords.shape[0]            # Make the y values cumulative (0.0-1.0)\n",
    "    return pos_count,cum_props"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f1aef47b",
   "metadata": {},
   "outputs": [],
   "source": [
    "def logistic_fn(params,x):\n",
    "    #print(params,x,y)\n",
    "    k,x0=params\n",
    "    return 1.0/(1+np.exp(-k*(x-x0)))\n",
    "def logistic_fn_err(params,x,y):\n",
    "    return np.sum ( ( logistic_fn(params,x)-y)**2 )"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "dc3dc3cb",
   "metadata": {},
   "outputs": [],
   "source": [
    "def bootstrap(data):\n",
    "    # Make a bootstrap sample from data (resample same size array)\n",
    "    idxs=np.random.randint(0,len(data),size=len(data),dtype='int')\n",
    "    return data[[idxs]]\n",
    "\n",
    "def bootstrap2(data,ndrop): # Drop N from the data\n",
    "    trues=np.array([True]*len(data))\n",
    "    idxs=np.random.randint(0,len(data),size=ndrop,dtype='int')\n",
    "    trues[[idxs]]=False\n",
    "    return data[[trues]]\n",
    "    \n",
    "# 95% confidence intervals. Just index into the sorted bootstrap values at right places\n",
    "def conf95(arr,p=0.05):\n",
    "    arr_sorted=np.sort(arr)\n",
    "    stats=(arr_sorted[int(len(arr)*p)], # lower\n",
    "            arr_sorted[len(arr)//2], # Middle value (median)\n",
    "            arr_sorted[int(len(arr)*(1.0-p))]) # upper\n",
    "    \n",
    "    str_stats='%0.2f-%0.2f'%(stats[0],stats[2])\n",
    "    return stats,str_stats    "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a2696087",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Original data\n",
    "pos_count,cum_props=make_stats(coords)\n",
    "counts,pos=np.histogram(compute_nn(coords),bins=21)\n",
    "plt.bar( pos[1:],counts/np.max(counts), alpha=0.2, color='b')\n",
    "plt.plot( pos_count[:,0], cum_props, color='k', lw=3 )\n",
    "\n",
    "# Fitted logistic\n",
    "res = scipy.optimize.minimize(logistic_fn_err, x0=(0.5,20), method='nelder-mead',\n",
    "               args=(pos_count[:,0],cum_props), options={'xatol': 1e-8, 'disp': True})\n",
    "xr=np.linspace(pos_count[0,0],pos_count[-1,0],100) # Smooth curve just for plotting\n",
    "plt.plot( xr, logistic_fn(res.x, xr), color='r', label='k=%0.2f\\n$x_0$=%0.2f'%(res.x[0],res.x[1]))\n",
    "plt.legend(loc='upper left')\n",
    "plt.legend(loc='upper left')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "40e095fe",
   "metadata": {},
   "outputs": [],
   "source": [
    "resamp=bootstrap(coords) # Boot strap is too extreme! (drops ~50 cones)\n",
    "pos_count,cum_props=make_stats(resamp)\n",
    "\n",
    "counts,pos=np.histogram(compute_nn(resamp),bins=21)\n",
    "plt.bar( pos[1:],counts/np.max(counts), alpha=0.2, color='b')\n",
    "\n",
    "# Actual cumulative plot:\n",
    "plt.plot( pos_count[:,0], cum_props, color='k', lw=3 )\n",
    "xr=np.linspace(pos_count[0,0],pos_count[-1,0],100) # Smooth curve just for plotting\n",
    "\n",
    "# Fitted logistic\n",
    "res = scipy.optimize.minimize(logistic_fn_err, x0=(0.5,20), method='nelder-mead',\n",
    "               args=(pos_count[:,0],cum_props), options={'xatol': 1e-8, 'disp': True})\n",
    "plt.plot( xr, logistic_fn(res.x, xr), color='r', label='k=%0.2f\\n$x_0$=%0.2f'%(res.x[0],res.x[1]))\n",
    "plt.legend(loc='upper left')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9d7284bc",
   "metadata": {},
   "outputs": [],
   "source": [
    "NUM_BOOTS=1000\n",
    "params=np.zeros((NUM_BOOTS,2))\n",
    "cones_included=np.zeros( NUM_BOOTS)\n",
    "for n in np.arange(NUM_BOOTS):\n",
    "    #resamp=bootstrap(coords)\n",
    "    resamp=bootstrap2(coords,np.random.randint(0,17)) # Randomly drop out ~ this # of cones\n",
    "    cones_included[n]=np.sum( np.array([np.sum( np.all( (coords1==resamp), axis=1 ) ) for coords1 in coords])>0 ) # How many original cones appear in the bootstrap (nonzero times)\n",
    "    pos_count,cum_props=make_stats(resamp)\n",
    "    res = scipy.optimize.minimize(logistic_fn_err, x0=(0.5,20), method='nelder-mead',\n",
    "                   args=(pos_count[:,0],cum_props), options={'xatol': 1e-8, 'disp': False})\n",
    "    params[n]=res.x"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "0aaaa319",
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.hist( cones_included )"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c1b15dea",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Try to make a good distribution of sizes to see whether that makes a difference\n",
    "size_for_cones_included_in_bootstrap=(cones_included-np.min(cones_included))\n",
    "size_for_cones_included_in_bootstrap /= np.max(size_for_cones_included_in_bootstrap)\n",
    "size_for_cones_included_in_bootstrap *= 100\n",
    "#plt.hist( size_for_cones_included_in_bootstrap)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "ec3d4b60",
   "metadata": {},
   "outputs": [],
   "source": [
    "df=pd.DataFrame(params,columns=[\"k\",\"$x_0$\"])\n",
    "axes=sns.jointplot(data=df,x='k',y='$x_0$',# c=[cm.Spectral(c1) for c1 in size_for_cones_included_in_bootstrap],\n",
    "       joint_kws=dict(s=size_for_cones_included_in_bootstrap,ec='k'),  alpha=0.6\n",
    "                  )\n",
    "\n",
    "ax=axes.ax_joint\n",
    "ax.axvline(np.mean(params[:,0]), color='r', ls='--')\n",
    "ax.axhline(np.mean(params[:,1]), color='r', ls='--')\n",
    "ax.text(1,1, \"k=%0.2f (%s)\\n$x_0$=%0.2f (%s)\"%(np.mean(params[:,0]), conf95(params[:,0])[1],\n",
    "                                          np.mean(params[:,1]), conf95(params[:,1])[1]), \n",
    "        transform=ax.transAxes, ha='right', va='top', size=20 # text Justified to top-right corner\n",
    "\n",
    "       )"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "31d33e1b",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
